# Branching & Merging
**Topic:** Git & Version Control

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** what a branch is and why it does not copy the entire codebase
- **Describe** the three merge scenarios: fast-forward, merge commit, and merge conflict
- **Interpret** a branch diagram and identify where branches diverge and where they rejoin

> **Tip:** Start by selecting **Fast-forward merge** from the dropdown and trace both branch lines before switching to **Merge conflict** to see how the diagram changes.

---
## How we got here

In *02: Git Basics* we learned the core commit workflow: edit, stage, commit. Every commit we made went directly to the `main` branch. Branches let you work in parallel: you can experiment, try a different model architecture, or fix a bug, all without touching the `main` branch that your teammates rely on. This is the concept most students struggle with, and the diagram in this notebook is the key to making it click.

---
## Why this matters for data science

Branches are the foundation of professional collaboration. Every company that uses Git follows some form of the feature branch workflow: new work happens on a separate branch, gets reviewed as a pull request, and merges into `main` only when it is ready. As a data scientist, you will create a branch for every new experiment, model version, or analysis. Knowing how merges work saves you from the confusion and panic that come with your first merge conflict.

---
## Try it yourself

In [ ]:
# Widget: A dropdown with three merge scenarios:
#   1. Fast-forward merge: feature branch is ahead of main; no divergence
#   2. Merge commit: both main and feature have new commits since branching
#   3. Merge conflict: both branches modified the same line in the same file
# For each scenario, the branch diagram updates to show:
#   - main branch as a horizontal line with commit circles
#   - feature branch diverging at the branch point
#   - The merge result (a merge commit node or a fast-forward line)
#   - For merge conflict: a red highlight on the conflicting commit with a panel
#     showing the conflict markers (<<<<<<< HEAD ... ======= ... >>>>>>> feature)
# Student should notice: a branch is just a pointer to a commit, not a copy of files;
#   fast-forward is the simplest merge and leaves no merge commit in the history

---
## What's happening?

A branch in Git is simply a lightweight, movable pointer to a specific commit. Creating a branch does not copy any files. It just creates a new label that starts pointing at the same commit you are on, then moves forward independently as you add new commits.

```
Before branching:
  main:  A ── B ── C  (HEAD → main)

After git checkout -b experiment:
  main:       A ── B ── C
  experiment:          C  (HEAD → experiment, points to same commit C)

After two commits on experiment:
  main:       A ── B ── C
  experiment:            C ── D ── E  (HEAD → experiment)
```

| Merge type | When it happens | What the history looks like |
|-----------|----------------|---------------------------|
| Fast-forward | Feature branch is strictly ahead of main; main has no new commits | A straight line; no merge commit created |
| Merge commit (3-way) | Both main and feature have new commits since branching | A diamond: two lines converge into a new merge commit |
| Merge conflict | Both branches modified the same line in the same file | Merge halts; Git marks the conflict; you resolve manually, then commit |

```bash
# The standard feature branch workflow
git checkout main          # start from main
git pull                   # make sure you have the latest
git checkout -b experiment/log-transform   # create a branch
# ... do work, make commits ...
git checkout main
git merge experiment/log-transform         # merge when ready
git branch -d experiment/log-transform     # clean up
```

### Resolving a merge conflict

When Git cannot automatically merge two changes, it inserts conflict markers into the file:

```python
<<<<<<< HEAD
income = df["income"].clip(upper=500_000)   # your version on main
=======
income = np.log1p(df["income"])              # the version on your branch
>>>>>>> experiment/log-transform
```

You edit the file to keep the version you want (or combine both), remove the markers, then `git add` and `git commit` to complete the merge. It sounds alarming; it is actually straightforward.

Return to the widget and step through the conflict scenario to see the markers and practice resolving them.

---
## Real-world example: Feature branch workflow on a model development project

The chart below shows the branch history of a real data science project over four weeks. Each row is a branch; the horizontal spans show how long each branch lived before merging back to main.

Notice:

- **Notice:** The `main` branch receives a merge roughly once per week; between merges it remains stable and deployable while experiments happen on separate branches
- **Notice:** Two branches overlap in time, meaning two team members were working in parallel without interfering with each other
- **Notice:** The `fix/timezone-bug` branch was short-lived by design; bug fixes are merged as quickly as possible so the fix reaches production without waiting for the experiment branches

> **Discussion question:** A teammate argues that branches add overhead and the team should just commit everything directly to `main`. They point out that a two-person team could coordinate by talking to each other. At what team size does the "just talk to each other" approach break down, and what is the first painful situation that would change your teammate's mind?

In [ ]:
# ── Branch lifetime diagram for a DS project over 4 weeks ────────────────────
# Each row shows a branch as a horizontal bar from branch creation to merge
branches = [
    ("main",                          0, 28, PALETTE["primary"]),
    ("feat/log-transform-income",     2, 10, PALETTE["secondary"]),
    ("feat/smote-oversampling",       5, 18, PALETTE["accent"]),
    ("fix/timezone-bug",              12, 15, "#C92A2A"),
    ("feat/merchant-category-feats",  16, 26, PALETTE["muted"]),
]

fig = go.Figure()
for name, start, end, color in branches:
    fig.add_trace(go.Scatter(
        x=[start, end], y=[name, name],
        mode="lines+markers",
        line=dict(color=color, width=8),
        marker=dict(
            size=[14, 14],
            symbol=["circle", "diamond"],
            color=color,
        ),
        name=name,
        hovertemplate=f"{name}<br>Day {start} → Day {end} ({end-start} days)<extra></extra>",
    ))
    if name != "main":
        fig.add_annotation(
            x=end, y=name,
            text=" ← merged",
            showarrow=False,
            font=dict(size=10, color=PALETTE["muted"]),
            xanchor="left",
        )

layout = base_layout(
    title="Branch Lifetimes — Data Science Project (4-week sprint)",
    xaxis_title="Project Day",
    yaxis_title="",
)
layout.update(xaxis=dict(range=[-1, 32]))
fig.update_layout(layout)
fig.show()

### Merge type comparison

| Merge type | When it happens | What the history looks like | When to use it |
|-----------|----------------|---------------------------|---------------|
| Fast-forward | Feature branch is ahead of main; no divergence | Straight line, no merge commit | Small solo changes; keeps history clean |
| Merge commit (--no-ff) | Both branches have new commits, or forced | Diamond: two lines meet at a merge commit | Team collaboration; preserves that a branch existed |
| Squash merge | Multiple feature commits collapsed into one | One clean commit on main | When feature branch history is messy |
| Rebase | Feature commits replayed on top of main | Linear history, no merge commit | Advanced; rewrites history: never on shared branches |
| Merge conflict | Same line changed on both branches | Merge halts; manual resolution required | Not a choice: it happens when changes collide |

---
## Key takeaway

> **A branch is a lightweight pointer to a commit, not a copy of the codebase; creating one is instant and free, which means you should make a new branch for every experiment, feature, and bug fix.**

---
*Next up: GitHub & Remote Collaboration — how to push your branches to the internet, share them with teammates, and contribute to shared projects through pull requests*